In [1]:
import os
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np

import catboost as cb
import lightgbm as lgb
import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

from mllabs import Experimenter, Connector, ColSelector, ProgressSessionLogger, TqdmProgressSession
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier
from mllabs.col import ohe_drop_first
from mllabs.collector import SHAPCollector, StackingCollector, ProbToLabel
from mllabs.filter import RandomFilter
from mllabs.processor import PolarsLoader, PandasConverter, TypeConverter, CatConverter, ExprProcessor, CrossFitTransformer
from mllabs.collector import MetricCollector, ModelAttrCollector

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import balanced_accuracy_score

2026-05-16 20:56:31.456192: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-16 20:56:31.491278: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-16 20:56:32.633690: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/sun9sun9/python312/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: Fu

In [2]:
data_path = Path('data')

dict_expr = {
    'Mulching_Used_n': (pl.col('Mulching_Used') == 'Yes').cast(pl.Int8),
    'Soil_25': (pl.col('Soil_Moisture') < 25).cast(pl.Int8),
    'Temp_30': (pl.col('Temperature_C') > 30).cast(pl.Int8),
    'Rain_300': (pl.col('Rainfall_mm') < 300).cast(pl.Int8),
    'Wind_10': (pl.col('Wind_Speed_kmh') > 10).cast(pl.Int8),
}

loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor(dict_expr=dict_expr),
)

df_train = loader.fit_transform([data_path / 'train.csv'])
df_test = loader.transform([data_path / 'test.csv'])

y = 'Irrigation_Need'
X_cat = ['Crop_Growth_Stage', 'Crop_Type', 'Irrigation_Type', 'Region', 'Season', 'Soil_Type', 'Water_Source']
X_num = ['Electrical_Conductivity', 'Field_Area_hectare', 'Humidity', 'Organic_Carbon', 'Previous_Irrigation_mm', 'Rainfall_mm', 
         'Soil_Moisture', 'Soil_pH', 'Sunlight_Hours', 'Temperature_C','Wind_Speed_kmh']
X_bin = ['Mulching_Used_n']
X_num_bin = ['Soil_25', 'Temp_30', 'Rain_300', 'Wind_10']
X_all = X_num + X_cat + X_bin + X_num_bin

y_repl = {'Low': 0, 'Medium': 1, 'High': 2}
y_weight = {'Low': 1.000000, 'Medium': 1.547291, 'High': 17.607549}
df_train = df_train.with_columns(
    **{
        y: pl.col(y).replace(y_repl).cast(pl.Int8),
        'sample_weight': pl.col(y).cast(pl.String).replace(y_weight).cast(pl.Float32)
    }
)

In [4]:
if not os.path.exists('exp/modeling'):
    skf1 = StratifiedKFold(5, random_state = 123, shuffle=True)
    sss2 = StratifiedShuffleSplit(n_splits = 1, random_state = 123, train_size = 0.9)
    ex = Experimenter.create(
        df_train, 'exp/modeling', sp = skf1, sp_v = sss2, splitter_params={'y': y}, 
        logger=ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession))
    ex.set_grp('clf', method = 'predict_proba', role = 'head', edges = {'y': [(None, y)], 'sample_weight': [(None, 'sample_weight')]})
    ex.set_grp(
        'xgb', parent = 'clf', processor=xgb.XGBClassifier, adapter=XGBoostAdapter(eval_mode='both'), 
        params={'random_state': 123, 'n_estimators': 10000, 'learning_rate': 0.05, 'enable_categorical': True, 'early_stopping_rounds': 50, 'eval_metric': 'mlogloss'})
    ex.add_collector(ModelAttrCollector('xgb_evals_results', Connector(processor=xgb.XGBClassifier), 'evals_result'))
    ex.set_grp(
        'lgb', parent = 'clf', processor=lgb.LGBMClassifier, adapter=LightGBMAdapter(eval_mode='both'), 
        params={'random_state': 123, 'n_estimators': 10000, 'learning_rate': 0.05, 'verbose': -1, 'early_stopping': {'stopping_rounds': 50, 'first_metric_only': True},'eval_metric': 'multi_logloss'})
    ex.add_collector(
        ModelAttrCollector('lgb_evals_results', Connector(processor=lgb.LGBMClassifier), 'evals_result'))
    ex.set_grp(
        'cb', parent = 'clf', processor=cb.CatBoostClassifier, adapter=CatBoostAdapter(eval_mode='valid'),
        params={'early_stopping_rounds': 50, 'verbose': 0, 'random_state': 123, 'cat_features': ColSelector(col_type='category')})
    ex.add_collector(
        ModelAttrCollector('cb_evals_results', Connector('_base$', processor=cb.CatBoostClassifier), 'evals_result'))
    ex.set_grp('nn', parent = 'clf', processor = NNClassifier, params = {'metrics': ['sparse_categorical_crossentropy'], 'early_stopping': 10})
    ex.add_collector(
        ModelAttrCollector('nn_evals', Connector(processor=NNClassifier), result_key='evals_result'))
    ex.set_grp('lr', parent='clf', processor=LogisticRegression)

    ex.set_grp('pre', role = 'stage', method='transform')
    ex.set_grp('pre_ft', role = 'stage', method='fit_transform', edges = {'y': [(None, y)]})
    ex.set_node('std', grp='pre', processor=StandardScaler, edges={'X': [(None, X_num)]})
    ex.set_node('ohe', grp='pre', processor=OneHotEncoder, edges={'X': [(None, X_cat)]}, params={'sparse_output': False})

    # For catboost to be considered as Categorical Type
    ex.set_node('X_num_bin2str', grp='pre', edges={'X': [(None, X_num_bin)]}, processor=TypeConverter, params={'to':'str'})
    ex.set_node('X_num_binstr2cat', grp='pre', edges={'X': [('X_num_bin2str', None)]}, processor=CatConverter)

    ex.set_node('X_lr_cf', grp='pre_ft', edges={'X': [('ohe', ('cap', ohe_drop_first, ['ohe__Crop_Growth_Stage.*'])), (None, X_bin), (None, X_num_bin)]}, 
              processor=CrossFitTransformer, params={'estimator':LogisticRegression()})
    ex.build()
    
    y_edges = {'y': [(None, y)]}
    ex.add_collector(
        MetricCollector('bAcc', Connector(edges = y_edges, role = 'head'), None, ProbToLabel(balanced_accuracy_score, var=y), include_train = True))
    ex.add_collector(
        StackingCollector(
            'stacking', Connector(edges=y_edges, role='head'),
            slice(0, -1), method='mean'
        ))
else:
    ex = Experimenter.load('exp/modeling', df_train, logger=ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession))

📁 Created directory: exp/modeling
Building 5 node(s)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Build complete: 5 node(s)


In [ ]:
ex.set_node('xgb_X_lr_cf', grp='xgb', edges={'X': [(None, X_num + X_cat + X_bin + X_num_bin), ('X_lr_cf', None)]}, params={'n_estimators': 10000, 'gpu': 'yes'})
ex.set_node('lgb_X_lr_cf', grp='lgb', edges={'X': [(None, X_num + X_cat + X_bin + X_num_bin), ('X_lr_cf', None)]}, params={'n_estimators': 10000, 'gpu': None})
ex.set_node('cb_X_lr_cf', grp='cb', edges={'X': [(None, X_num + X_num_bin + X_cat + ['Mulching_Used']), ('X_lr_cf', None)]}, params={'n_estimators': 10000, 'gpu': 'yes'})
ex.set_node('nn_lr_cf', grp='nn', edges={'X': [('std', None), ('ohe', ohe_drop_first), (None, X_bin), (None, X_num_bin), ('X_lr_cf', slice(0, -1))]}, params={'epochs': 200, 'gpu': 'yes'})
ex.exp(finalize=False, n_jobs=2, gpu_id_list=[0])

In [ ]:
ex.add_trainer('trainer')
ex.trainers['trainer'].select_head(['xgb_X_lr_cf', 'lgb_X_lr_cf', 'cb_X_lr_cf', 'nn_lr_cf'])
ex.trainers['trainer'].train(n_jobs=2, gpu_id_list=[0])

In [12]:
infer = ex.trainers['trainer'].to_inferencer()

In [24]:
df_stacking = ex.get_collector('stacking').get_dataset()
df_stacking

lgb_X_lr_cf__Irrigation_Need_0,lgb_X_lr_cf__Irrigation_Need_1,cb_X_lr_cf__Irrigation_Need_0,cb_X_lr_cf__Irrigation_Need_1,nn_lr_cf__Irrigation_Need_0,nn_lr_cf__Irrigation_Need_1,xgb_X_lr_cf__Irrigation_Need_0,xgb_X_lr_cf__Irrigation_Need_1,Irrigation_Need
f64,f64,f64,f64,f64,f64,f64,f64,i8
0.996795,0.003181,0.998129,0.001871,0.991757,0.008243,0.996309,0.003684,0
0.997116,0.002871,0.998354,0.001646,0.997239,0.002761,0.999434,0.000566,0
0.997862,0.002124,0.99857,0.00143,0.993995,0.006005,0.999409,0.000591,0
0.996277,0.003695,0.997451,0.002549,0.996373,0.003627,0.996647,0.003352,0
0.999965,0.000034,0.999933,0.000067,0.999844,0.000156,0.999998,0.000002,0
…,…,…,…,…,…,…,…,…
0.996166,0.003833,0.993987,0.006013,0.997185,0.002815,0.985872,0.014127,0
0.999999,0.000001,0.999867,0.000133,0.999855,0.000145,0.999994,0.000006,0
3.8098e-7,0.989637,0.000001,0.957334,0.000025,0.797473,7.5304e-7,0.986183,1


In [51]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

clf_lr = LogisticRegression(C = 1000)
X_stk = [i for i in df_stacking.columns if i != y]
scores_stk = cross_val_score(clf_lr, df_stacking[X_stk], df_stacking[y], scoring = 'balanced_accuracy', cv = skf1)
scores_stk.mean()

np.float64(0.9664529444996017)

In [52]:
clf_lr.fit(df_stacking[X_stk], df_stacking[y])

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1000
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mu

In [75]:
prd = pd.Series(
    clf_lr.predict(infer.process(df_test, slice(0, -1))[X_stk]), index = pd.Index(df_test['id'], name = 'id'), name = y
).map({v:k for k, v in y_repl.items()})

In [76]:
prd.to_csv('data/Submission.csv')

In [78]:
# !kaggle competitions submit -c playground-series-s6e4 -f data/Submission.csv -m "1"

100%|██████████████████████████████████████| 3.13M/3.13M [00:02<00:00, 1.39MB/s]
Successfully submitted to Predicting Irrigation Need